# Module 21: Testing & Debugging
## Lesson: Writing Tests for ML Pipelines

In this lesson you will learn the fundamentals of testing Python code with pytest,
mocking external dependencies, debugging training loops, and setting up continuous
integration. Every concept is demonstrated with ML-specific examples.

In [ ]:
import pytest
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from unittest.mock import Mock, patch, PropertyMock
import logging
import sys

print("All imports successful")

### 1. pytest Basics: Assert, Fixtures, Parametrization

pytest is the most widely used testing framework in the Python ML ecosystem.
It uses plain `assert` statements, supports reusable fixtures, and enables
running the same test with multiple inputs via parametrization.

In [ ]:
def normalize_features(df, cols):
    """Min-max normalize specified columns."""
    result = df.copy()
    for col in cols:
        min_val = result[col].min()
        max_val = result[col].max()
        if max_val == min_val:
            result[col] = 0.0
        else:
            result[col] = (result[col] - min_val) / (max_val - min_val)
    return result


def test_normalize_features():
    data = pd.DataFrame({"age": [20, 30, 40], "income": [30000, 50000, 70000]})
    result = normalize_features(data, ["age", "income"])
    assert result["age"].min() == pytest.approx(0.0)
    assert result["age"].max() == pytest.approx(1.0)
    assert result["income"].min() == pytest.approx(0.0)
    assert result["income"].max() == pytest.approx(1.0)
    print("test_normalize_features PASSED")


test_normalize_features()


# Parametrized test example
@pytest.mark.parametrize("col,expected_min,expected_max", [
    ("age", 0.0, 1.0),
    ("income", 0.0, 1.0),
])
def test_normalize_col(col, expected_min, expected_max):
    data = pd.DataFrame({"age": [20, 30, 40], "income": [30000, 50000, 70000]})
    result = normalize_features(data, [col])
    assert result[col].min() == pytest.approx(expected_min)
    assert result[col].max() == pytest.approx(expected_max)

### 2. Fixtures with conftest.py

Fixtures provide reusable test data. Shared fixtures go in conftest.py
and are automatically discovered by pytest. Scope controls fixture lifetime:
- `function` (default): created/destroyed per test
- `class`: once per test class
- `module`: once per module
- `session`: once per test run

In [ ]:
# This would normally go in conftest.py
import pytest


@pytest.fixture
def sample_house_data():
    return pd.DataFrame({
        "sqft": [1000, 1500, 2000, 2500, 3000],
        "bedrooms": [2, 3, 3, 4, 4],
        "bathrooms": [1, 2, 2, 3, 3],
        "price": [200000, 300000, 400000, 500000, 600000],
    })


@pytest.fixture(scope="module")
def trained_model():
    from sklearn.ensemble import RandomForestRegressor
    X = np.random.rand(50, 5)
    y = np.random.rand(50)
    model = RandomForestRegressor(n_estimators=10, random_state=42)
    model.fit(X, y)
    return model


def test_fixture_usage(sample_house_data):
    assert sample_house_data.shape == (5, 4)
    assert list(sample_house_data.columns) == ["sqft", "bedrooms", "bathrooms", "price"]
    print("test_fixture_usage PASSED")


def test_trained_model_shape(trained_model):
    preds = trained_model.predict(np.random.rand(3, 5))
    assert preds.shape == (3,)
    print("test_trained_model_shape PASSED")


test_fixture_usage(sample_house_data())
test_trained_model_shape(trained_model())

### 3. Mocking ML Models and External Dependencies

Mocking replaces real objects with fake ones during testing. This is essential
for ML projects where model training is slow or external APIs may be unavailable.
We use `unittest.mock` and pytest's `monkeypatch`.

In [ ]:
def train_and_evaluate(model_class, X_train, y_train, X_test, y_test):
    model = model_class()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    from sklearn.metrics import mean_squared_error
    mse = mean_squared_error(y_test, predictions)
    return model, mse


# Using Mock to avoid actual training
mock_model = Mock()
mock_model.predict.return_value = np.array([0.5, 0.6, 0.7])
mock_model.fit.return_value = None

# Simulate what train_and_evaluate does
X_test_fake = np.random.rand(3, 2)
y_test_fake = np.array([0.4, 0.5, 0.6])
predictions = mock_model.predict(X_test_fake)
print(f"Mock predictions: {predictions}")
mock_model.predict.assert_called_once_with(X_test_fake)
print("Mock was called correctly")


# Using patch as a decorator
def mock_training_expensive():
    with patch("sklearn.ensemble.RandomForestRegressor.fit") as mock_fit:
        mock_fit.return_value = None
        from sklearn.ensemble import RandomForestRegressor
        model = RandomForestRegressor()
        model.fit(np.random.rand(10, 3), np.random.rand(10))
        print("fit() was mocked — no actual training occurred")
        mock_fit.assert_called_once()


mock_training_expensive()

### 4. Debugging with pdb and Logging

Two complementary approaches: interactive debugging with pdb/ipdb and
structured logging. Use pdb for stepping through code, logging for
tracking execution over time.

In [ ]:
# Setting up logging for debugging
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(name)s | %(levelname)s | %(message)s",
    stream=sys.stdout,
)
logger = logging.getLogger("training_loop")


def debug_training_loop(epochs=5):
    """Simulate a training loop with debug logging."""
    loss = 1.0
    for epoch in range(epochs):
        loss *= 0.5
        noise = np.random.randn() * 0.01
        loss += noise
        logger.debug(f"Epoch {epoch+1}/{epochs}: loss = {loss:.6f}")
        if np.isnan(loss):
            logger.error(f"NaN detected at epoch {epoch+1}")
            # In real code: import pdb; pdb.set_trace()
            break
    logger.info(f"Final loss: {loss:.6f}")
    return loss


final_loss = debug_training_loop(10)
print(f"Training completed with loss={final_loss:.6f}")

### 5. Test Coverage with pytest-cov

Coverage measures which lines of code are executed during tests.
For ML projects, pay special attention to coverage in:
- Data preprocessing functions
- Feature engineering logic
- Error handling paths
- Edge cases in transformations

In [ ]:
# Coverage analysis is typically run from the command line:
# pytest --cov=src/ --cov-report=term-missing --cov-report=html

print("Run this in terminal:")
print("  pytest --cov=. --cov-report=term-missing")
print()
print("Example coverage report output:")
print("  Name                 Stmts   Miss  Cover   Missing")
print("  src/data_loader.py      25      2    92%   34-35")
print("  src/features.py         30      5    83%   12,18-21")
print("  src/model.py            40     10    75%   22-25,40-45")
print("  TOTAL                    95     17    82%")

### 6. CI Basics with GitHub Actions

Continuous Integration runs your tests automatically on every push.
For ML projects, CI should also check data integrity and model performance.

In [ ]:
# GitHub Actions workflow (.github/workflows/tests.yml)
yaml_config = """
name: CI
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: "3.10"
      - run: pip install -r requirements.txt
      - run: pip install pytest pytest-cov
      - run: pytest --cov=src/ --cov-fail-under=80
"""
print("GitHub Actions workflow:")
print(yaml_config)

### 7. TDD Principles for ML

Test-Driven Development: Red → Green → Refactor

For ML projects, TDD helps catch subtle bugs:
1. **RED**: Write a test for a new feature (e.g., imputation strategy)
2. **GREEN**: Implement the feature minimally
3. **REFACTOR**: Clean up while tests pass

Key ML testing patterns:
- Test data shape invariants (input → output dimensions)
- Test value ranges (predictions in [0,1] for probability)
- Test deterministic behavior (same seed → same result)
- Test error handling (empty data, missing columns)

In [ ]:
# TDD Example: Test first, then implement

def test_impute_missing():
    data = pd.DataFrame({"x": [1.0, np.nan, 3.0, np.nan, 5.0]})
    result = impute_with_median(data, "x")
    assert result["x"].isnull().sum() == 0
    assert result["x"].iloc[1] == 3.0  # median of [1, 3, 5]
    assert result["x"].iloc[3] == 3.0
    print("test_impute_missing design — now implement impute_with_median")


def impute_with_median(df, column):
    """Fill missing values in column with median."""
    result = df.copy()
    median_val = result[column].median()
    result[column] = result[column].fillna(median_val)
    return result


test_impute_missing()
print("TDD cycle complete: test written, then implementation passed")

### Summary

In this lesson you covered:
- pytest fundamentals: assert, fixtures, parametrization
- Organizing tests with conftest.py and fixture scoping
- Mocking ML models and external dependencies
- Debugging training loops with pdb and logging
- Measuring test coverage with pytest-cov
- Setting up GitHub Actions CI for ML projects
- Applying TDD principles to ML pipeline development

**Next**: Complete the exercises to practice these concepts hands-on.